# 03 — From Tiling to Crease Pattern (Shrink–Rotate)

The **shrink–rotate** construction (SRG, also called *reciprocal figures*) is the algorithm at the heart of `eucare`. Given a tiling:

1. Each face is *shrunk* toward its centroid.
2. The shrunk copies are *rotated* about their centroids.
3. The gaps between shrunk faces are filled with reciprocal triangles — these are the *creases*.

The result is a flat-foldable crease pattern whose folded state approximates a downscaled copy of the original tiling sitting rigidly on top of itself.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    plotting,
    reciprocal_figures,
    rendering,
)


def plot_g(G, ax=None, color='black', linewidth=1.0):
    """Draw the edges of a half-edge graph G on `ax` (or the current axes)."""
    if ax is None:
        ax = plt.gca()
    lines = np.array([
        [G.geometry.to_euclidean(h.orig['pos']),
         G.geometry.to_euclidean(h.dest['pos'])]
        for h in G.halfedges_representing_edges()
    ])
    plotting.plot_lines(lines, ax=ax, colors=color, linewidths=linewidth)
    plotting.set_equal_aspect(ax)
    ax.axis('off')


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


## Build a square tiling and its SRG

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_g(G,   ax=axes[0]); axes[0].set_title('input tiling')
plot_g(SRG, ax=axes[1]); axes[1].set_title('shrink-rotate CP')
plt.tight_layout(); plt.show()


## Why z-order matters

SRG needs a partial order on faces — it tells the algorithm which way each face rotates. The simplest assignment is BFS distance from a central face, which is what the `srg_pipeline` helper does.

Different orderings give different (but equivalent up to symmetry) patterns; this is a useful knob when designing for fabrication.

## Standard render preset

For consistent visual style across the docs we expose `eucare.rendering.CREASE_PATTERN_PRESET` and the colour constants `MOUNTAIN_COLOR`, `VALLEY_COLOR`, `FLAT_COLOR`.

In [ ]:
print(rendering.CREASE_PATTERN_PRESET)
print(rendering.MOUNTAIN_COLOR, rendering.VALLEY_COLOR, rendering.FLAT_COLOR)


## What's next

- [`04_Folding_and_Overlap`](04_Folding_and_Overlap.ipynb) — turn this CP into a folded state with mountain/valley assignment.
- [`Better reciprocal Figures`](Better%20reciprocal%20Figures.ipynb) is the legacy companion with more advanced examples.